In [44]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y mediapipe opencv-python opencv-python-headless opencv-contrib-python

!pip install --no-cache-dir \
    numpy==1.26.4 \
    protobuf==5.29.6 \
    opencv-contrib-python==4.11.0.86 \
    tqdm

!pip install --no-cache-dir --no-deps mediapipe==0.10.21

Found existing installation: mediapipe 0.10.21
Uninstalling mediapipe-0.10.21:
  Successfully uninstalled mediapipe-0.10.21
Found existing installation: opencv-python 4.13.0.92
Uninstalling opencv-python-4.13.0.92:
  Successfully uninstalled opencv-python-4.13.0.92
Found existing installation: opencv-contrib-python 4.11.0.86
Uninstalling opencv-contrib-python-4.11.0.86:
  Successfully uninstalled opencv-contrib-python-4.11.0.86
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 229.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 128.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.6
    Uninstalling numpy-2.4.6:
      Successfully uninstalled numpy-2.4.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ultra

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 167.6 MB/s eta 0:00:00


In [ ]:
!pip install -q ultralytics

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 0.10.21 requires sounddevice>=0.4.4, which is not installed.
albucore 0.0.24 requires opencv-python-headless>=4.9.0.80, which is not installed.
albumentations 2.0.8 requires opencv-python-headless>=4.9.0.80, which is not installed.
mediapipe 0.10.21 requires numpy<2, but you have numpy 2.4.6 which is incompatible.
mediapipe 0.10.21 requires protobuf<5,>=4.25.3, but you have protobuf 5.29.6 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.6 which is incompatible.


In [ ]:
!pip install -q onnx onnxscript onnxruntime-gpu

In [ ]:
import os
import cv2
import json
import torch
import mediapipe as mp
import numpy as np

import torch.nn as nn

from PIL import Image
from torchvision import models
from torchvision import transforms

from ultralytics import YOLO
from collections import defaultdict

import onnx
import onnxscript
import onnxruntime as ort

print("ONNX version :", onnx.__version__)
print("ONNX Runtime version :", ort.__version__)
print("Available Providers :", ort.get_available_providers())


ONNX version : 1.22.0
ONNX Runtime version : 1.26.0
Available Providers : ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']


In [ ]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

MODEL_PATH = "/content/drive/MyDrive/salpim_project/02_models/fusion_mediapipe_pose_best.pth"

NUM_CLASSES = 10
SEQ_LEN = 90
NUM_FRAMES = 16

In [ ]:
LABEL_MAP = {
    0:"A010",
    1:"A011",
    2:"A016",
    3:"A018",
    4:"A023",
    5:"A031",
    6:"A035",
    7:"A041",
    8:"A053",
    9:"A054"
}

MediaPipe 설정

In [ ]:
mp_pose = mp.solutions.pose

pose_model = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    smooth_landmarks=True,
    enable_segmentation=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)
# yolo 모델 로드
yolo_model = YOLO("yolov8n.pt")

YOLO 후 Pose 추출 함수

In [ ]:
def extract_pose_single(person_crop):

    if person_crop is None or person_crop.size == 0:
        return None

    rgb = cv2.cvtColor(person_crop, cv2.COLOR_BGR2RGB)

    result = pose_model.process(rgb)

    if result.pose_landmarks is None:
        return None

    landmarks = result.pose_landmarks.landmark

    SELECTED_LANDMARKS = [
        0,
        11, 12,
        13, 14,
        15, 16,
        23, 24,
        25, 26,
        27, 28,
        5, 2, 7, 8
    ]

    coords = []

    for idx in SELECTED_LANDMARKS:

        lm = landmarks[idx]

        coords.append([
            lm.x,
            lm.y,
            lm.z
        ])

    coords = np.array(coords, dtype=np.float32)

    # -------------------------
    # Hip Normalize
    # -------------------------

    left_hip = coords[7]
    right_hip = coords[8]

    hip_center = (left_hip + right_hip) / 2

    coords = coords - hip_center

    # -------------------------
    # Body Scale Normalize
    # -------------------------

    left_shoulder = coords[1]
    right_shoulder = coords[2]

    shoulder_center = (
        left_shoulder + right_shoulder
    ) / 2

    body_scale = np.linalg.norm(
        shoulder_center
    )

    if body_scale > 1e-6:
        coords = coords / body_scale

    return coords

RGB extractor

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

resnet = models.resnet18(
    weights="IMAGENET1K_V1"
)

resnet.fc = nn.Identity()

resnet.to(DEVICE)
resnet.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

ST-GCN

In [ ]:
import torch
import torch.nn as nn
import numpy as np


def build_coco17_adjacency(num_joints=17):
    """
    COCO 17 keypoint 기준 관절 연결 그래프 생성
    0 nose
    1 left_eye, 2 right_eye
    3 left_ear, 4 right_ear
    5 left_shoulder, 6 right_shoulder
    7 left_elbow, 8 right_elbow
    9 left_wrist, 10 right_wrist
    11 left_hip, 12 right_hip
    13 left_knee, 14 right_knee
    15 left_ankle, 16 right_ankle
    """

    edges = [
        (0, 1), (0, 2),
        (1, 3), (2, 4),
        (5, 6),
        (5, 7), (7, 9),
        (6, 8), (8, 10),
        (5, 11), (6, 12),
        (11, 12),
        (11, 13), (13, 15),
        (12, 14), (14, 16),
    ]

    A = np.eye(num_joints, dtype=np.float32)

    for i, j in edges:
        A[i, j] = 1
        A[j, i] = 1

    # degree normalization: D^(-1/2) A D^(-1/2)
    D = np.sum(A, axis=1)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(D + 1e-6))
    A_norm = D_inv_sqrt @ A @ D_inv_sqrt

    return torch.tensor(A_norm, dtype=torch.float32)


class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, dropout=0.5):
        super().__init__()

        self.register_buffer("A", A)

        # spatial graph convolution
        self.gcn = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

        # temporal convolution
        self.tcn = nn.Sequential(
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=(9, 1),
                stride=(stride, 1),
                padding=(4, 0)
            ),
            nn.BatchNorm2d(out_channels),
            nn.Dropout(dropout)
        )

        if in_channels == out_channels and stride == 1:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=(stride, 1)
                ),
                nn.BatchNorm2d(out_channels)
            )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (N, C, T, V)

        # graph convolution
        # 각 관절 feature를 인접 관절 feature와 섞음
        x_g = torch.einsum("nctv,vw->nctw", x, self.A)
        x_g = self.gcn(x_g)

        # temporal convolution + residual
        out = self.tcn(x_g) + self.residual(x)

        return self.relu(out)


class STGCNActionClassifier(nn.Module):
    def __init__(self, in_channels, num_classes, num_joints=17, dropout=0.5):
        super().__init__()

        A = build_coco17_adjacency(num_joints)

        self.data_bn = nn.BatchNorm1d(in_channels * num_joints)

        self.layer1 = STGCNBlock(in_channels, 64, A, stride=1, dropout=dropout)
        self.layer2 = STGCNBlock(64, 64, A, stride=1, dropout=dropout)
        self.layer3 = STGCNBlock(64, 128, A, stride=2, dropout=dropout)
        self.layer4 = STGCNBlock(128, 128, A, stride=1, dropout=dropout)
        self.layer5 = STGCNBlock(128, 256, A, stride=2, dropout=dropout)

        self.fc = nn.Identity() ###### 수정

    def forward(self, x):
        # x: (N, C, T, V)
        N, C, T, V = x.shape

        # BatchNorm을 위해 (N, C, T, V) -> (N, C*V, T)
        x = x.permute(0, 1, 3, 2).contiguous()
        x = x.view(N, C * V, T)
        x = self.data_bn(x)

        # 다시 (N, C, T, V)
        x = x.view(N, C, V, T)
        x = x.permute(0, 1, 3, 2).contiguous()

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.layer5(x)

        # global average pooling over time and joints
        x = x.mean(dim=[2, 3])

        logits = self.fc(x)

        return logits


Fusion Model

In [ ]:
class RGBSkeletonFusion(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.rgb_fc = nn.Linear(512, 256)

        self.skeleton_model = STGCNActionClassifier(
            in_channels=6,
            num_classes=256,
            num_joints=17
        )

        self.classifier = nn.Sequential(
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, rgb_feat, skeleton):

        sk_feat = self.skeleton_model(skeleton)

        rgb_feat = self.rgb_fc(rgb_feat)

        fused = (
            0.4 * rgb_feat +
            0.6 * sk_feat
        )

        out = self.classifier(fused)

        return out

모델 로드

In [ ]:
model = RGBSkeletonFusion(
    num_classes=10
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=DEVICE
    )
)

model.to(DEVICE)
model.eval()

print("Fusion Model Loaded")

Fusion Model Loaded


학습과 동일한 설정

In [ ]:
# 학습과 동일한 관절 선택

SELECTED_LANDMARKS = [
    0,
    11,12,
    13,14,
    15,16,
    23,24,
    25,26,
    27,28,
    5,2,7,8
]

In [ ]:
# 학습 때 사용한 normalize함수
def normalize_pose(landmarks):

    coords = landmarks[:, :3]

    left_hip = coords[23]
    right_hip = coords[24]

    hip_center = (
        left_hip + right_hip
    ) / 2

    left_shoulder = coords[11]
    right_shoulder = coords[12]

    shoulder_center = (
        left_shoulder + right_shoulder
    ) / 2

    body_scale = np.linalg.norm(
        shoulder_center - hip_center
    )

    if body_scale < 1e-6:
        body_scale = 1.0

    coords = coords - hip_center
    coords = coords / body_scale

    coords = coords[
        SELECTED_LANDMARKS
    ]

    return coords

In [ ]:
def add_velocity(sequence_xyz):

    velocity = np.zeros_like(
        sequence_xyz,
        dtype=np.float32
    )

    velocity[1:] = (
        sequence_xyz[1:]
        - sequence_xyz[:-1]
    )

    return np.concatenate(
        [sequence_xyz, velocity],
        axis=-1
    )

In [ ]:
def pad_or_sample_sequence(
    sequence,
    target_len=90
):

    T = len(sequence)

    if T > target_len:

        idx = np.linspace(
            0,
            T-1,
            target_len
        ).astype(int)

        return sequence[idx]

    if T < target_len:

        pad = np.repeat(
            sequence[-1:],
            target_len-T,
            axis=0
        )

        return np.concatenate(
            [sequence,pad],
            axis=0
        )

    return sequence

ONNX 세션

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(
    "fusion_best.onnx",
    providers=[
        "CUDAExecutionProvider",
        "CPUExecutionProvider"
    ]
)

print(session.get_providers())

['CUDAExecutionProvider', 'CPUExecutionProvider']


In [ ]:
import onnxruntime as ort

sess = ort.InferenceSession("fusion_best.onnx")
print(sess.get_inputs())
print(sess.get_outputs())

In [45]:
import shutil

shutil.copy(
    "fusion_best.onnx",
    "/content/drive/MyDrive/salpim_project/02_models/fusion_best.onnx"
)

print("Drive 저장 완료")

Drive 저장 완료


ByteTrack 추론 함수

track id 마다 buffer = 90이 되는 순간 fusion추론 하도록

In [ ]:
from collections import defaultdict
import time

track_buffers = defaultdict(list)

def predict_video_bytetrack(video_path):

    cap = cv2.VideoCapture(video_path)

    frame_idx = 0

    # =========================
    # FPS 측정 시작
    # =========================
    start_time = time.time()
    total_frames = 0

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frame_idx += 1
        total_frames += 1 ###

        results = yolo_model.track(
            frame,
            persist=True,
            tracker="bytetrack.yaml",
            classes=[0],
            conf=0.5,
            verbose=False
        )

        if len(results) == 0:
            continue

        boxes = results[0].boxes

        if boxes.id is None:
            continue

        for box, track_id in zip(
            boxes.xyxy,
            boxes.id
        ):

            track_id = int(track_id.item())

            x1, y1, x2, y2 = map(
                int,
                box.tolist()
            )

            h, w = frame.shape[:2]

            x1 = max(0, x1)
            y1 = max(0, y1)

            x2 = min(w, x2)
            y2 = min(h, y2)

            crop = frame[
                y1:y2,
                x1:x2
            ]


            pose = extract_pose_single(crop)
            ##################
            if pose is None:

                print(
                    f"Track {track_id} pose fail"
                )

                continue
            #####################
            track_buffers[track_id].append(
                pose
            )

            track_rgb_frames[track_id].append(
                crop.copy()
            )

            if (len(track_buffers[track_id]) >= 90 and len(track_buffers[track_id]) % 10 == 0):

                # ------------------
                # Skeleton
                # ------------------

                sequence_xyz = np.array(
                    track_buffers[track_id][-90:],
                    dtype=np.float32
                )

                skeleton = add_velocity(
                    sequence_xyz
                )

                skeleton = torch.tensor(
                    skeleton,
                    dtype=torch.float32
                )

                skeleton = skeleton.permute(
                    2, 0, 1
                )

                skeleton = skeleton.unsqueeze(0)

                skeleton = skeleton.to(DEVICE)

                # ------------------
                # RGB Feature
                # ------------------

                rgb_feat = extract_rgb_feature(
                    track_rgb_frames[track_id][-90:]
                )

                rgb_feat = torch.tensor(
                    rgb_feat,
                    dtype=torch.float32
                )

                rgb_feat = rgb_feat.unsqueeze(0)

                rgb_feat = rgb_feat.to(DEVICE)

                # ------------------
                # Fusion
                # ------------------

                # output = model(
                #     rgb_feat,
                #     skeleton
                # )

                # pred = output.argmax(1).item()

                # ------------------
                # ONNX Fusion
                # ------------------

                rgb_np = rgb_feat.detach().cpu().numpy()

                skeleton_np = skeleton.detach().cpu().numpy()

                output = session.run(
                    None,
                    {
                        "rgb_feat": rgb_np,
                        "skeleton": skeleton_np
                    }
                )

                pred = np.argmax(
                    output[0],
                    axis=1
                ).item()

                action = LABEL_MAP[pred]

                track_results[track_id] = action

                print(
                    f"Track {track_id} | 행동 : {action}"
                )

            #print(
            #    f"Frame={frame_idx} "
                f"Track={track_id} "
                f"Buffer={len(track_buffers[track_id])}"

            #)

    cap.release()

    # =========================
    # FPS 계산
    # =========================

    elapsed_time = time.time() - start_time

    fps = total_frames / elapsed_time

    print("\n========================")
    print(" Full Pipeline FPS")
    print("========================")
    print(f"Total Frames : {total_frames}")
    print(f"Elapsed Time : {elapsed_time:.2f} sec")
    print(f"FPS : {fps:.2f}")
    print("========================\n")

    return track_buffers

Track Buffer 생성

In [ ]:
track_buffers = defaultdict(list)

track_rgb_frames = defaultdict(list)

track_results = {}

RGB feature 추출

In [ ]:
@torch.no_grad()
def extract_rgb_feature(frames):

    if len(frames) >= 16:

        idx = np.linspace(
            0,
            len(frames)-1,
            16,
            dtype=int
        )

        frames = [
            frames[i]
            for i in idx
        ]

    else:

        while len(frames) < 16:
            frames.append(frames[-1])

    tensors = []

    for frame in frames:

        img = Image.fromarray(
            cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )
        )

        tensors.append(
            transform(img)
        )

    tensors = torch.stack(
        tensors
    ).to(DEVICE)

    feat = resnet(tensors)

    feat = feat.mean(dim=0)

    return feat.cpu().numpy()

Skeleton 생성

In [ ]:
def extract_skeleton(frames):

    sequence_xyz = []

    for frame in frames:

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        result = pose_model.process(rgb)

        if result.pose_landmarks is None:
            continue

        landmarks = []

        for lm in result.pose_landmarks.landmark:

            landmarks.append([
                lm.x,
                lm.y,
                lm.z,
                lm.visibility
            ])

        landmarks = np.array(
            landmarks,
            dtype=np.float32
        )

        coords = normalize_pose(
            landmarks
        )

        sequence_xyz.append(coords)

    sequence_xyz = np.array(
        sequence_xyz,
        dtype=np.float32
    )

    sequence_6d = add_velocity(
        sequence_xyz
    )

    sequence_6d = pad_or_sample_sequence(
        sequence_6d,
        90
    )

    return sequence_6d

최종 추론

In [ ]:
@torch.no_grad()
def predict_video(video_path):

    cap = cv2.VideoCapture(video_path)

    frames = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

    cap.release()

    rgb_feat = extract_rgb_feature(
        frames
    )

    skeleton = extract_skeleton(
        frames
    )

    rgb_feat = torch.tensor(
        rgb_feat,
        dtype=torch.float32
    ).unsqueeze(0).to(DEVICE)

    skeleton = torch.tensor(
        skeleton,
        dtype=torch.float32
    )

    skeleton = skeleton.permute(
        2,0,1
    )

    skeleton = skeleton.unsqueeze(0)
    skeleton = skeleton.to(DEVICE)

    output = model(
        rgb_feat,
        skeleton
    )

    pred = output.argmax(1).item()

    return LABEL_MAP[pred]

실행

In [ ]:
video_path = "/content/drive/MyDrive/salpim_project/00_data/RGB_selected/P229/A031_P229_G002_H070.mp4"

results = predict_video_bytetrack(
    video_path
)



Track 45 pose fail
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011
Track 46 | 행동 : A011


KeyboardInterrupt: 

Fusion 모델 FPS 측정

In [ ]:
import time
import torch
import cv2

video_path = "/content/drive/MyDrive/salpim_project/05_merged_videos/merged_video(1).mp4"

# ----------------------------------
# 영상 전처리 (1회만 수행)
# ----------------------------------

cap = cv2.VideoCapture(video_path)

frames = []

while True:

    ret, frame = cap.read()

    if not ret:
        break

    frames.append(frame)

cap.release()

rgb_feat = extract_rgb_feature(frames)

skeleton = extract_skeleton(frames)

rgb_feat = torch.tensor(
    rgb_feat,
    dtype=torch.float32
).unsqueeze(0).to(DEVICE)

skeleton = torch.tensor(
    skeleton,
    dtype=torch.float32
)

skeleton = skeleton.permute(
    2, 0, 1
)

skeleton = skeleton.unsqueeze(0)
skeleton = skeleton.to(DEVICE)

# ----------------------------------
# Warm-up
# ----------------------------------

for _ in range(20):
    _ = model(
        rgb_feat,
        skeleton
    )

# ----------------------------------
# Fusion 모델 추론속도 측정
# ----------------------------------

N = 100

start = time.time()

for _ in range(N):

    output = model(
        rgb_feat,
        skeleton
    )

end = time.time()

avg_time = (end - start) / N

fps = 1 / avg_time

print(f"Average Inference Time : {avg_time*1000:.3f} ms")
print(f"FPS : {fps:.2f}")

ONNX Export

In [ ]:
import torch

model.eval()

dummy_rgb = torch.randn(
    1, 512
).to(DEVICE)

dummy_skeleton = torch.randn(
    1, 6, 90, 17
).to(DEVICE)

torch.onnx.export(
    model,
    (dummy_rgb, dummy_skeleton),

    "fusion_best.onnx",

    export_params=True,

    opset_version=17,

    do_constant_folding=True,

    input_names=[
        "rgb_feat",
        "skeleton"
    ],

    output_names=[
        "output"
    ],

    dynamic_axes={
        "rgb_feat": {
            0: "batch_size"
        },

        "skeleton": {
            0: "batch_size"
        },

        "output": {
            0: "batch_size"
        }
    }
)

print("ONNX Export Complete")

In [ ]:
session = ort.InferenceSession(
    "fusion_best.onnx",
    providers=[
        "CUDAExecutionProvider",
        "CPUExecutionProvider"
    ]
)
print(session.get_providers())
rgb_np = rgb_feat.detach().cpu().numpy()

skeleton_np = skeleton.detach().cpu().numpy()

ONNX FPS 측정

In [ ]:
import time

# warm-up
for _ in range(20):
    _ = session.run(
        None,
        {
            "rgb_feat": rgb_np,
            "skeleton": skeleton_np
        }
    )

N = 100

start = time.time()

for _ in range(N):

    _ = session.run(
        None,
        {
            "rgb_feat": rgb_np,
            "skeleton": skeleton_np
        }
    )

end = time.time()

avg_time = (end - start) / N

fps = 1 / avg_time

print(f"Average Inference Time : {avg_time*1000:.3f} ms")
print(f"FPS : {fps:.2f}")